# Fase 5 — Verificação de Overlap Cross-Dataset

Detecta textos repetidos, URLs repetidas, near-duplicates e conflitos de label entre:
- **V2**: dataset_final_treino_v2 (balanced preferido, fallback full)
- **FakeTrueBR**: faketruebr_curated_full
- **FakeBR**: fakebr_curated_text_normalized

**Regras**: Apenas leitura e análise. Nenhum arquivo existente é modificado.

In [1]:
# ── Célula 1: Imports e PROJECT_ROOT ──────────────────────────────────────────
import os, sys, glob, re, unicodedata
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

# Detecta PROJECT_ROOT de forma robusta
def find_project_root(markers=("CLAUDE.md", ".git", "dados", "src")):
    candidates = [Path.cwd(), Path(__file__).parent if "__file__" in dir() else Path.cwd()]
    candidates += list(Path.cwd().parents)
    for p in candidates:
        if any((p / m).exists() for m in markers):
            return p
    raise RuntimeError("PROJECT_ROOT não encontrado")

PROJECT_ROOT = find_project_root()
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
OUTPUT_DIR = PROJECT_ROOT / "dados" / "pipeline_datasets_academicos" / "final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"TIMESTAMP   : {TIMESTAMP}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")

PROJECT_ROOT: C:\Users\offan\Desktop\ml-checkai
TIMESTAMP   : 2026-05-25_00-27-10
OUTPUT_DIR  : C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\final


In [2]:
# ── Célula 2: Localizar arquivos mais recentes ────────────────────────────────
def latest_file(pattern_glob, base=PROJECT_ROOT):
    """Retorna o arquivo mais recente que bate com o glob."""
    matches = sorted(Path(base).glob(pattern_glob))
    if not matches:
        return None
    # Ordena por timestamp no nome (string sort funciona para YYYY-MM-DD_HH-MM-SS)
    return sorted(matches, key=lambda p: p.stem)[-1]

# V2: prefere balanced, fallback para full
f_v2 = latest_file("dados/dataset_unificado/final/dataset_final_treino_v2_balanced_*.csv")
if f_v2 is None:
    f_v2 = latest_file("dados/dataset_unificado/final/dataset_final_treino_v2_full_*.csv")
    print("[AVISO] V2 balanced não encontrado — usando full")

f_ftbr = latest_file("dados/pipeline_datasets_academicos/curated/faketruebr/faketruebr_curated_full_*.csv")
f_fbr  = latest_file("dados/pipeline_datasets_academicos/curated/fakebr/fakebr_curated_text_normalized_*.csv")

for label, path in [("V2", f_v2), ("FakeTrueBR", f_ftbr), ("FakeBR_TextNorm", f_fbr)]:
    if path is None:
        raise FileNotFoundError(f"Arquivo não encontrado para: {label}")
    size_mb = path.stat().st_size / 1_048_576
    print(f"{label:20s}: {path.name}  ({size_mb:.1f} MB)")

V2                  : dataset_final_treino_v2_balanced_2026-05-19_00-48-15.csv  (0.2 MB)
FakeTrueBR          : faketruebr_curated_full_2026-05-24_20-01-57.csv  (6.9 MB)
FakeBR_TextNorm     : fakebr_curated_text_normalized_2026-05-24_20-28-04.csv  (40.2 MB)


In [3]:
# ── Célula 3: Carregar datasets ───────────────────────────────────────────────
REQUIRED_COLS = {"id_registro", "texto_principal", "label"}

def load_and_validate(path, name):
    df = pd.read_csv(path, low_memory=False)
    missing = REQUIRED_COLS - set(df.columns)
    if missing:
        raise ValueError(f"{name}: colunas ausentes: {missing}")
    print(f"{name:20s}: {len(df):>6} linhas | colunas: {list(df.columns)}")
    return df

df_v2   = load_and_validate(f_v2,   "V2")
df_ftbr = load_and_validate(f_ftbr, "FakeTrueBR")
df_fbr  = load_and_validate(f_fbr,  "FakeBR_TextNorm")

V2                  :    508 linhas | colunas: ['id_registro', 'texto_principal', 'label', 'label_detalhe', 'pipeline_origem', 'portal_origem', 'origem_texto', 'origem_qualidade', 'tamanho_chars', 'data_publicacao', 'url_origem']
FakeTrueBR          :   3095 linhas | colunas: ['id_registro', 'texto_principal', 'label', 'label_detalhe', 'pipeline_origem', 'portal_origem', 'origem_texto', 'origem_qualidade', 'tamanho_chars', 'data_publicacao', 'url_origem', 'fonte_dataset', 'referencia_dataset', 'faixa_tamanho']


FakeBR_TextNorm     :   7182 linhas | colunas: ['id_registro', 'texto_principal', 'label', 'label_detalhe', 'pipeline_origem', 'portal_origem', 'origem_texto', 'origem_qualidade', 'tamanho_chars', 'data_publicacao', 'url_origem', 'fonte_dataset', 'referencia_dataset', 'faixa_tamanho', 'tamanho_chars_original', 'texto_principal_modelo', 'tamanho_chars_modelo', 'faixa_tamanho_modelo', 'texto_truncado', 'limite_truncamento']


In [4]:
# ── Célula 4: Distribuição de labels ─────────────────────────────────────────
def label_dist(df, name):
    dist = df["label"].value_counts()
    print(f"\n{name}")
    for lbl, cnt in dist.items():
        print(f"  {lbl}: {cnt} ({cnt/len(df)*100:.1f}%)")
    return dist

dist_v2   = label_dist(df_v2,   "V2")
dist_ftbr = label_dist(df_ftbr, "FakeTrueBR")
dist_fbr  = label_dist(df_fbr,  "FakeBR_TextNorm")


V2
  0: 254 (50.0%)
  1: 254 (50.0%)

FakeTrueBR
  0: 1752 (56.6%)
  1: 1343 (43.4%)

FakeBR_TextNorm
  0: 3600 (50.1%)
  1: 3582 (49.9%)


In [5]:
# ── Célula 5: Normalização de texto para comparação ───────────────────────────
def normalizar(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"\s+", " ", s).strip()
    return s

print("Normalizando textos...")
df_v2["texto_norm"]   = df_v2["texto_principal"].map(normalizar)
df_ftbr["texto_norm"] = df_ftbr["texto_principal"].map(normalizar)
df_fbr["texto_norm"]  = df_fbr["texto_principal"].map(normalizar)
print("Normalização concluída.")

Normalizando textos...


Normalização concluída.


In [6]:
# ── Célula 6: Overlap exato por texto ────────────────────────────────────────
def overlap_exato(dfA, nameA, dfB, nameB):
    setA = set(dfA["texto_norm"].dropna())
    setB = set(dfB["texto_norm"].dropna())
    inter = setA & setB
    print(f"{nameA} vs {nameB}: {len(inter)} overlaps exatos ({len(setA)} | {len(setB)} únicos)")
    if not inter:
        return pd.DataFrame()
    rowsA = dfA[dfA["texto_norm"].isin(inter)][["id_registro","texto_principal","label","texto_norm"]].copy()
    rowsA["dataset_origem"] = nameA
    rowsB = dfB[dfB["texto_norm"].isin(inter)][["id_registro","texto_principal","label","texto_norm"]].copy()
    rowsB["dataset_origem"] = nameB
    merged = pd.concat([rowsA, rowsB], ignore_index=True).sort_values("texto_norm")
    return merged

ov_v2_ftbr = overlap_exato(df_v2, "V2", df_ftbr, "FakeTrueBR")
ov_v2_fbr  = overlap_exato(df_v2, "V2", df_fbr,  "FakeBR_TextNorm")
ov_ft_fb   = overlap_exato(df_ftbr, "FakeTrueBR", df_fbr, "FakeBR_TextNorm")

V2 vs FakeTrueBR: 0 overlaps exatos (507 | 3091 únicos)
V2 vs FakeBR_TextNorm: 0 overlaps exatos (507 | 7182 únicos)
FakeTrueBR vs FakeBR_TextNorm: 0 overlaps exatos (3091 | 7182 únicos)


In [7]:
# ── Célula 7: Overlap por url_origem ─────────────────────────────────────────
def overlap_url(dfA, nameA, dfB, nameB):
    if "url_origem" not in dfA.columns or "url_origem" not in dfB.columns:
        print(f"{nameA} vs {nameB}: url_origem ausente em um dos datasets — pulando")
        return pd.DataFrame()
    urlA = set(dfA["url_origem"].dropna().astype(str).str.strip())
    urlB = set(dfB["url_origem"].dropna().astype(str).str.strip())
    urlA.discard(""); urlB.discard("")
    inter = urlA & urlB
    print(f"{nameA} vs {nameB} [URL]: {len(inter)} URLs repetidas ({len(urlA)} | {len(urlB)} com URL)")
    if not inter:
        return pd.DataFrame()
    rowsA = dfA[dfA["url_origem"].astype(str).isin(inter)][["id_registro","label","url_origem"]].copy()
    rowsA["dataset_origem"] = nameA
    rowsB = dfB[dfB["url_origem"].astype(str).isin(inter)][["id_registro","label","url_origem"]].copy()
    rowsB["dataset_origem"] = nameB
    return pd.concat([rowsA, rowsB], ignore_index=True).sort_values("url_origem")

url_v2_ftbr = overlap_url(df_v2, "V2", df_ftbr, "FakeTrueBR")
url_v2_fbr  = overlap_url(df_v2, "V2", df_fbr,  "FakeBR_TextNorm")
url_ft_fb   = overlap_url(df_ftbr, "FakeTrueBR", df_fbr, "FakeBR_TextNorm")

V2 vs FakeTrueBR [URL]: 3 URLs repetidas (502 | 2905 com URL)
V2 vs FakeBR_TextNorm [URL]: 0 URLs repetidas (502 | 7156 com URL)
FakeTrueBR vs FakeBR_TextNorm [URL]: 3 URLs repetidas (2905 | 7156 com URL)


In [8]:
# ── Célula 8: Conflitos de label ─────────────────────────────────────────────
def conflitos_label(dfA, nameA, dfB, nameB):
    """Textos iguais com labels diferentes entre os dois datasets."""
    inter = set(dfA["texto_norm"]) & set(dfB["texto_norm"])
    if not inter:
        print(f"{nameA} vs {nameB} [conflitos]: sem overlap — 0 conflitos")
        return pd.DataFrame()
    mapA = dfA.drop_duplicates("texto_norm").set_index("texto_norm")["label"]
    mapB = dfB.drop_duplicates("texto_norm").set_index("texto_norm")["label"]
    rows = []
    for txt in inter:
        la = mapA.get(txt)
        lb = mapB.get(txt)
        if la != lb:
            rows.append({"texto_norm": txt, f"label_{nameA}": la, f"label_{nameB}": lb})
    print(f"{nameA} vs {nameB} [conflitos]: {len(rows)} conflito(s) de label")
    return pd.DataFrame(rows)

conf_v2_ftbr = conflitos_label(df_v2, "V2", df_ftbr, "FakeTrueBR")
conf_v2_fbr  = conflitos_label(df_v2, "V2", df_fbr,  "FakeBR_TextNorm")
conf_ft_fb   = conflitos_label(df_ftbr, "FakeTrueBR", df_fbr, "FakeBR_TextNorm")

all_conflicts = pd.concat([conf_v2_ftbr, conf_v2_fbr, conf_ft_fb], ignore_index=True)

V2 vs FakeTrueBR [conflitos]: sem overlap — 0 conflitos
V2 vs FakeBR_TextNorm [conflitos]: sem overlap — 0 conflitos
FakeTrueBR vs FakeBR_TextNorm [conflitos]: sem overlap — 0 conflitos


In [9]:
# ── Célula 9: Near-duplicates via TF-IDF ─────────────────────────────────────
# Executado somente se o total de textos for viável (≤ 80k)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

TFIDF_THRESHOLD = 0.92   # similaridade mínima para considerar near-duplicate
TFIDF_MAX_ROWS  = 80_000 # limite para manter custo computacional razoável

nd_results = []

def near_duplicates_pair(dfA, nameA, dfB, nameB, threshold=TFIDF_THRESHOLD):
    total = len(dfA) + len(dfB)
    if total > TFIDF_MAX_ROWS:
        print(f"{nameA} vs {nameB}: {total} linhas > limite {TFIDF_MAX_ROWS} — pulando TF-IDF")
        return pd.DataFrame()
    textsA = dfA["texto_norm"].fillna("").tolist()
    textsB = dfB["texto_norm"].fillna("").tolist()
    tfidf = TfidfVectorizer(max_features=30_000, sublinear_tf=True)
    tfidf.fit(textsA + textsB)
    matA = tfidf.transform(textsA)
    matB = tfidf.transform(textsB)
    rows = []
    # Processamento em batches para evitar explosão de memória
    batch = 500
    for start in range(0, len(textsA), batch):
        end = min(start + batch, len(textsA))
        sims = linear_kernel(matA[start:end], matB)
        idxA, idxB = np.where(sims >= threshold)
        for ia, ib in zip(idxA, idxB):
            real_ia = start + ia
            sim_val = sims[ia, ib]
            tA = textsA[real_ia]; tB = textsB[ib]
            if tA == tB:  # overlap exato já capturado
                continue
            rows.append({
                f"idx_{nameA}": dfA.iloc[real_ia]["id_registro"],
                f"idx_{nameB}": dfB.iloc[ib]["id_registro"],
                f"label_{nameA}": dfA.iloc[real_ia]["label"],
                f"label_{nameB}": dfB.iloc[ib]["label"],
                "similaridade": round(float(sim_val), 4),
                "par": f"{nameA}|{nameB}",
            })
    print(f"{nameA} vs {nameB} [near-dup]: {len(rows)} par(es) com sim ≥ {threshold}")
    return pd.DataFrame(rows)

nd_v2_ftbr = near_duplicates_pair(df_v2, "V2", df_ftbr, "FakeTrueBR")
nd_v2_fbr  = near_duplicates_pair(df_v2, "V2", df_fbr,  "FakeBR_TextNorm")
nd_ft_fb   = near_duplicates_pair(df_ftbr, "FakeTrueBR", df_fbr, "FakeBR_TextNorm")

all_near_dups = pd.concat([nd_v2_ftbr, nd_v2_fbr, nd_ft_fb], ignore_index=True)

V2 vs FakeTrueBR [near-dup]: 0 par(es) com sim ≥ 0.92


V2 vs FakeBR_TextNorm [near-dup]: 0 par(es) com sim ≥ 0.92


FakeTrueBR vs FakeBR_TextNorm [near-dup]: 5 par(es) com sim ≥ 0.92


In [10]:
# ── Célula 10: Montar relatório principal ─────────────────────────────────────
records = []

def add_record(par, tipo, qtd, detalhe=""):
    records.append({"par": par, "tipo": tipo, "quantidade": qtd, "detalhe": detalhe})

for (nameA, nameB), ov, url_ov, conf, nd in [
    (("V2", "FakeTrueBR"),     ov_v2_ftbr, url_v2_ftbr, conf_v2_ftbr, nd_v2_ftbr),
    (("V2", "FakeBR_TextNorm"),ov_v2_fbr,  url_v2_fbr,  conf_v2_fbr,  nd_v2_fbr),
    (("FakeTrueBR", "FakeBR_TextNorm"), ov_ft_fb, url_ft_fb, conf_ft_fb, nd_ft_fb),
]:
    par = f"{nameA} × {nameB}"
    n_ov   = len(ov["texto_norm"].unique()) if not ov.empty else 0
    n_url  = len(url_ov["url_origem"].unique()) if not url_ov.empty else 0
    n_conf = len(conf)
    n_nd   = len(nd)
    add_record(par, "overlap_exato_texto",    n_ov)
    add_record(par, "overlap_url",             n_url)
    add_record(par, "conflito_label",          n_conf)
    add_record(par, "near_duplicate_tfidf",    n_nd, f"threshold={TFIDF_THRESHOLD}")

df_report = pd.DataFrame(records)
print(df_report.to_string(index=False))

                         par                 tipo  quantidade        detalhe
             V2 × FakeTrueBR  overlap_exato_texto           0               
             V2 × FakeTrueBR          overlap_url           3               
             V2 × FakeTrueBR       conflito_label           0               
             V2 × FakeTrueBR near_duplicate_tfidf           0 threshold=0.92
        V2 × FakeBR_TextNorm  overlap_exato_texto           0               
        V2 × FakeBR_TextNorm          overlap_url           0               
        V2 × FakeBR_TextNorm       conflito_label           0               
        V2 × FakeBR_TextNorm near_duplicate_tfidf           0 threshold=0.92
FakeTrueBR × FakeBR_TextNorm  overlap_exato_texto           0               
FakeTrueBR × FakeBR_TextNorm          overlap_url           3               
FakeTrueBR × FakeBR_TextNorm       conflito_label           0               
FakeTrueBR × FakeBR_TextNorm near_duplicate_tfidf           5 threshold=0.92

In [11]:
# ── Célula 11: Salvar relatórios com timestamp ────────────────────────────────
report_path = OUTPUT_DIR / f"overlap_report_{TIMESTAMP}.csv"
df_report.to_csv(report_path, index=False, encoding="utf-8-sig")
print(f"Salvo: {report_path}")

if not all_conflicts.empty:
    conf_path = OUTPUT_DIR / f"overlap_label_conflicts_{TIMESTAMP}.csv"
    all_conflicts.to_csv(conf_path, index=False, encoding="utf-8-sig")
    print(f"Salvo: {conf_path}")
else:
    print("Sem conflitos de label — arquivo não gerado.")

if not all_near_dups.empty:
    nd_path = OUTPUT_DIR / f"overlap_near_duplicates_{TIMESTAMP}.csv"
    all_near_dups.to_csv(nd_path, index=False, encoding="utf-8-sig")
    print(f"Salvo: {nd_path}")
else:
    print("Sem near-duplicates — arquivo não gerado.")

Salvo: C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\final\overlap_report_2026-05-25_00-27-10.csv
Sem conflitos de label — arquivo não gerado.
Salvo: C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\final\overlap_near_duplicates_2026-05-25_00-27-10.csv


In [12]:
# ── Célula 12: Recomendação para V3 ──────────────────────────────────────────
total_overlaps = df_report[df_report["tipo"] == "overlap_exato_texto"]["quantidade"].sum()
total_conflicts = df_report[df_report["tipo"] == "conflito_label"]["quantidade"].sum()
total_nd = df_report[df_report["tipo"] == "near_duplicate_tfidf"]["quantidade"].sum()

print("\n" + "="*60)
print("RESUMO FINAL — VERIFICAÇÃO DE OVERLAP CROSS-DATASET")
print("="*60)
print(f"\nArquivos analisados:")
print(f"  V2              : {f_v2.name}")
print(f"  FakeTrueBR      : {f_ftbr.name}")
print(f"  FakeBR_TextNorm : {f_fbr.name}")
print(f"\nTotais por dataset:")
print(f"  V2              : {len(df_v2):>6} registros")
print(f"  FakeTrueBR      : {len(df_ftbr):>6} registros")
print(f"  FakeBR_TextNorm : {len(df_fbr):>6} registros")
print(f"\nDistribuição de labels:")
for name, dist in [("V2", dist_v2), ("FakeTrueBR", dist_ftbr), ("FakeBR_TextNorm", dist_fbr)]:
    print(f"  {name}: " + " | ".join(f"{k}={v}" for k,v in dist.items()))
print(f"\nOverlap exato (texto normalizado)  : {int(total_overlaps)} pares")
print(f"Conflitos de label                 : {int(total_conflicts)}")
print(f"Near-duplicates TF-IDF ≥{TFIDF_THRESHOLD}      : {int(total_nd)}")
print(f"\nRelatório salvo em                 : {report_path.name}")
print()

# Recomendação
print("RECOMENDAÇÃO PARA MONTAGEM DA V3:")
if total_overlaps == 0 and total_conflicts == 0 and total_nd == 0:
    print("  ✓ Nenhum overlap, conflito ou near-duplicate detectado.")
    print("  → Os três datasets podem ser concatenados diretamente na V3 sem risco de contaminação.")
elif total_conflicts > 0:
    print(f"  ✗ {int(total_conflicts)} conflito(s) de label detectado(s).")
    print("  → Revisar overlap_label_conflicts antes de montar V3. Remover duplicatas conflitantes.")
elif total_overlaps > 0 or total_nd > 0:
    print(f"  ⚠ {int(total_overlaps)} overlap(s) exato(s) e {int(total_nd)} near-duplicate(s).")
    print("  → Deduplicar por texto_norm antes de concatenar. Conflitos de label ausentes — baixo risco.")
print("="*60)


RESUMO FINAL — VERIFICAÇÃO DE OVERLAP CROSS-DATASET

Arquivos analisados:
  V2              : dataset_final_treino_v2_balanced_2026-05-19_00-48-15.csv
  FakeTrueBR      : faketruebr_curated_full_2026-05-24_20-01-57.csv
  FakeBR_TextNorm : fakebr_curated_text_normalized_2026-05-24_20-28-04.csv

Totais por dataset:
  V2              :    508 registros
  FakeTrueBR      :   3095 registros
  FakeBR_TextNorm :   7182 registros

Distribuição de labels:
  V2: 0=254 | 1=254
  FakeTrueBR: 0=1752 | 1=1343
  FakeBR_TextNorm: 0=3600 | 1=3582

Overlap exato (texto normalizado)  : 0 pares
Conflitos de label                 : 0
Near-duplicates TF-IDF ≥0.92      : 5

Relatório salvo em                 : overlap_report_2026-05-25_00-27-10.csv

RECOMENDAÇÃO PARA MONTAGEM DA V3:
  ⚠ 0 overlap(s) exato(s) e 5 near-duplicate(s).
  → Deduplicar por texto_norm antes de concatenar. Conflitos de label ausentes — baixo risco.
